# Module 6.4 — Code Quality & Linting

### Python Mastery · Track 6: Testing, Tooling & DevOps

Readable, consistent code is easier to maintain and review. Tools enforce this automatically: **linters** flag likely mistakes and style issues, **formatters** rewrite code into a consistent layout, and **pre-commit hooks** run these checks before code is committed. This module covers PEP 8, the modern tooling, and automated enforcement.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- The tools operate on files, so the examples write a `.py` file with `%%writefile` and run the tool with `!` to show real output, including before-and-after formatting.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. State the purpose of PEP 8 and the Zen of Python.
2. Run a linter (`ruff`) to find style issues and likely bugs.
3. Auto-format code with `black` and sort imports with `isort`.
4. Read a tool's diagnostics and apply automatic fixes.
5. Describe how pre-commit hooks enforce standards.

**Prerequisites:** Tracks 1 to 5.

---

## Part 1 · PEP 8 and the Zen of Python

**PEP 8** is Python's official style guide: it recommends four-space indentation, `lowercase_with_underscores` for functions and variables, `CapWords` for classes, spaces around operators, and a line-length limit, among much else. Consistent style is not pedantry; it reduces the effort of reading code and lets reviewers focus on substance.

The **Zen of Python** (PEP 20) captures the language's design philosophy in a set of aphorisms. It is built into the interpreter.

In [ ]:
import this   # prints the Zen of Python, a summary of the language's values

## Part 2 · Linting with `ruff`

A **linter** analyses code without running it, reporting style violations and likely errors: unused imports, undefined names, shadowed variables, and more. `ruff` is a fast, modern linter that consolidates the checks of many older tools. We write a file with several issues and let `ruff` report them.

In [ ]:
%%writefile messy_lint.py
import os
import sys


def calculate(x):
    unused_variable = 42
    result = x*2
    return result


print(calculate(5))

In [ ]:
# ruff reports each issue with a code (for example F401 = imported but unused).
!ruff check messy_lint.py

Many issues can be fixed automatically. Running `ruff check --fix` removes the unused imports and applies other safe corrections, leaving you to review only what needs judgement.

In [ ]:
# Apply ruff's automatic fixes, then show the cleaned-up file.
!ruff check --fix messy_lint.py
print("--- file after ruff --fix ---")
print(open("messy_lint.py").read())

## Part 3 · Auto-Formatting with `black`

A **formatter** rewrites code into one canonical layout, so the whole team's code looks the same and formatting never needs discussion in review. `black` is the most widely used Python formatter; it is deliberately opinionated and offers few options. We write deliberately untidy code and let `black` reformat it.

In [ ]:
%%writefile messy_format.py
def greet(name,greeting='Hello'):
    return greeting+', '+name
x = {  'a':1,'b':2,   'c':3}
result=[i*i for i in range(10) if i%2==0]
print(greet('Asha'),x,result)

In [ ]:
# First show what black would change (a diff), without modifying the file.
!black --diff messy_format.py

In [ ]:
# Now apply the formatting and show the result.
!black messy_format.py 2>/dev/null
print("--- file after black ---")
print(open("messy_format.py").read())

Notice how `black` standardised the spacing around operators and commas, the quote style, and the layout, all without changing behaviour. Because the rules are fixed, every file formatted with `black` looks consistent.

## Part 4 · Sorting Imports with `isort`

Imports are easier to scan when grouped and ordered consistently: standard library first, then third-party packages, then local modules, each group alphabetised. `isort` does this automatically, complementing the formatter.

In [ ]:
%%writefile messy_imports.py
import sys
from collections import defaultdict
import os
import json
from pathlib import Path


print("modules imported")

In [ ]:
# isort reorders and groups the imports.
!isort messy_imports.py
print("--- file after isort ---")
print(open("messy_imports.py").read())

## Part 5 · Enforcing Standards with Pre-Commit Hooks

Tools only help if they are actually run. A **pre-commit hook** runs chosen checks automatically whenever you commit, so badly formatted or failing code never enters the repository. The popular `pre-commit` framework is configured with a small YAML file listing the hooks to run. The file below is a typical configuration; in a real repository you would install it with `pre-commit install`, after which the hooks run on every commit.

In [ ]:
%%writefile .pre-commit-config.yaml
# A typical pre-commit configuration. In a real project, run:
#   pip install pre-commit
#   pre-commit install
# after which these checks run automatically on every `git commit`.
repos:
  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.6.0
    hooks:
      - id: ruff            # lint
        args: [--fix]
      - id: ruff-format     # format
  - repo: https://github.com/pre-commit/mirrors-mypy
    rev: v1.11.0
    hooks:
      - id: mypy            # type-check

In [ ]:
# We display the configuration to confirm it was written. (Running the hooks
# requires a git repository, which is beyond a single notebook.)
print(open(".pre-commit-config.yaml").read())

The payoff is that quality checks become automatic and unavoidable: every contributor's code is linted, formatted, and type-checked before it is committed, so the repository stays consistent without anyone having to remember to run the tools by hand. The same checks typically run again in continuous integration (Module 6.6) as a safety net.

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Linting a small file (Easy)

In [ ]:
%%writefile ex1_lint.py
import math
x = 5
print(x)

In [ ]:
# math is imported but never used; ruff flags it.
!ruff check ex1_lint.py

### Example 2 — Formatting a small file (Easy)

In [ ]:
%%writefile ex1_format.py
def f( a,b ):
    return a+b
print(f(1,2))

In [ ]:
!black ex1_format.py 2>/dev/null
print(open("ex1_format.py").read())

### Example 3 — Lint, then auto-fix (Medium)

In [ ]:
%%writefile ex3.py
import os
import json


def process(data):
    temp = json.dumps(data)
    return temp


print(process({"a": 1}))

In [ ]:
print("--- before fix (os is unused) ---")
!ruff check ex3.py
!ruff check --fix ex3.py
print("--- after fix ---")
print(open("ex3.py").read())

### Example 4 — Sorting imports (Medium)

In [ ]:
%%writefile ex4.py
import requests
import os
from mypackage import helper
import sys
print("ok")

In [ ]:
# isort groups standard library, third-party, and local imports separately.
!isort ex4.py 2>/dev/null
print(open("ex4.py").read())

### Example 5 — A full clean-up pipeline (Difficult)

In [ ]:
%%writefile ex5.py
import sys
import os
def compute(values):
    unused=0
    return sum( v*v for v in values )
result=compute([1,2,3])
print(result)

In [ ]:
print("=== step 1: sort imports ===")
!isort ex5.py 2>/dev/null
print("=== step 2: format ===")
!black ex5.py 2>/dev/null
print("=== step 3: lint and auto-fix ===")
!ruff check --fix ex5.py
print("=== final file ===")
print(open("ex5.py").read())

### Example 6 — Interpreting ruff diagnostics (Difficult)

In [ ]:
%%writefile ex6.py
def handler(items):
    result = []
    for i in items:
        result.append(i)
    unused = "never read"
    return result


x = handler([1, 2, 3])
print(x)

In [ ]:
# ruff reports the unused variable with a rule code and a short explanation.
!ruff check ex6.py
print("--- ruff can explain a rule code, for example F841 (assigned but never used) ---")

---

## Exercises

Write each file with `%%writefile`, then run the relevant tool with `!`.

**Exercise 1 (Easy).** Write a file that imports `os` but never uses it, then run `ruff check` and observe the unused-import warning.

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !ruff check cell here


**Exercise 2 (Easy).** Write a poorly spaced function (for example `def add(a,b):return a+b`), then run `black` on it and print the formatted result.

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !black + print cell here


**Exercise 3 (Medium).** Write a file with unsorted imports (a mix of standard library and a third-party name), run `isort`, and print the reordered file.

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !isort + print cell here


**Exercise 4 (Medium).** Write a file containing an unused variable inside a function, run `ruff check --fix`, and print the file afterwards to see whether the issue was removed.

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !ruff + print cell here


**Exercise 5 (Difficult).** Write an untidy file with unsorted imports, bad spacing, and an unused import. Apply `isort`, then `black`, then `ruff check --fix` in sequence, and print the final cleaned file.

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your pipeline + print cell here


---

## Solutions

**Solution 1**

In [ ]:
%%writefile sol1.py
import os
print("hello")

In [ ]:
!ruff check sol1.py

**Solution 2**

In [ ]:
%%writefile sol2.py
def add(a,b):return a+b
print(add(2,3))

In [ ]:
!black sol2.py 2>/dev/null
print(open("sol2.py").read())

**Solution 3**

In [ ]:
%%writefile sol3.py
import sys
import requests
import os
print("ok")

In [ ]:
!isort sol3.py 2>/dev/null
print(open("sol3.py").read())

**Solution 4**

In [ ]:
%%writefile sol4.py
def f():
    unused = 123
    return 1


print(f())

In [ ]:
!ruff check --fix sol4.py
print(open("sol4.py").read())

**Solution 5**

In [ ]:
%%writefile sol5.py
import sys
import json
import os
def go(data):
    extra=0
    return json.dumps(data)
print(go({"a":1}))

In [ ]:
!isort sol5.py 2>/dev/null
!black sol5.py 2>/dev/null
!ruff check --fix sol5.py
print(open("sol5.py").read())

---

## Summary and Key Points

- PEP 8 is the official style guide (indentation, naming, spacing, line length); consistent style lowers the effort of reading and reviewing code. The Zen of Python (PEP 20, via `import this`) states the language's values.
- A linter such as `ruff` analyses code without running it, flagging unused imports, likely bugs, and style issues, each with a rule code; `--fix` applies safe corrections automatically.
- A formatter such as `black` rewrites code into one canonical layout, removing formatting from review discussions; `isort` groups and orders imports consistently.
- Read diagnostics by their rule codes and apply automatic fixes where safe, reviewing the rest.
- Pre-commit hooks run these checks on every commit so substandard code never enters the repository; the same checks usually run again in continuous integration.

### Next module: 6.5 — Packaging & Distribution

The next module covers turning code into an installable package: virtual environments, dependencies, `pyproject.toml`, building a wheel, and publishing.